In [ ]:
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, root_mean_squared_log_error, r2_score

import numpy as np

df = pd.read_csv("/content/train_weights.csv")

X = df.drop("MSE", axis=1)
y = np.log1p(df.MSE)

# Определение категориальных признаков
categorical_features_indices = np.where(X.dtypes != np.float64)[0]



model = CatBoostRegressor(iterations=2000,
                          learning_rate=0.03,
                          depth=6,
                          loss_function='RMSE',
                          verbose=False, # Отключаем вывод в процессе обучения
                          cat_features=categorical_features_indices)

# Обучение модели
model.fit(X, y)

# Предсказание на тестовых данных
predictions = model.predict(X)

# Оценка производительности модели
rmsle = root_mean_squared_log_error(y, predictions)
print(f"root_mean_squared_log_error: {rmsle}")
print(f"r2 {r2_score(y, predictions)}")
print(100*max(min((0.3 - rmsle) / 0.1, 1), 0))

In [ ]:
X_test = pd.read_csv("/content/test_weights.csv")

In [ ]:
predictions = model.predict(X_test)
X_test["MSE"] = np.expm1(predictions)

X_test.to_json('answers', orient='records')

In [ ]:
import json
with open('answers') as f:
  obja = json.load(f)

print(json.dumps(obja, indent=4))